# 105. Construct Binary Tree from Preorder and Inorder Traversal

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** tree, dfs, recursion, hash-table, divide-and-conquer
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/)

Given two integer arrays `preorder` and `inorder` where `preorder` is the
preorder traversal of a binary tree and `inorder` is the inorder traversal of
the **same** tree, construct and return *the binary tree*.

---

### Example 1

```
Input:  preorder = [3, 9, 20, 15, 7], inorder = [9, 3, 15, 20, 7]
Output: [3, 9, 20, null, null, 15, 7]
```

```
      3
     / \
    9   20
        / \
      15   7
```

### Example 2

```
Input:  preorder = [-1], inorder = [-1]
Output: [-1]
```

---

### Constraints

- `1 <= preorder.length <= 3000`
- `inorder.length == preorder.length`
- `-3000 <= preorder[i], inorder[i] <= 3000`
- `preorder` and `inorder` consist of **unique** values.
- Each value of `inorder` also appears in `preorder`.
- `preorder` is **guaranteed** to be the preorder traversal of the tree.
- `inorder` is **guaranteed** to be the inorder traversal of the tree.

## Before you write anything

Every tree problem so far handed you a tree and asked for a list. This one is
the **reverse direction**: two lists in, a tree out. Answer these on paper.

**1.** Preorder is *root, left, right* - you wrote it in `preorderPrint` for
#114. So without looking at anything else: which element of `preorder` is
**guaranteed** to be the root of the whole tree?

**2.** Inorder is *left, root, right*. Take Example 1: the root you found in
question 1 sits somewhere inside `inorder = [9, 3, 15, 20, 7]`. Find it. Now
say precisely what the elements **to its left** are, and what the elements
**to its right** are. (This is the whole problem. Everything else is
bookkeeping.)

**3.** From question 2 you know the left subtree has some count `k` of nodes.
Preorder is *root, then the ENTIRE left subtree, then the entire right
subtree* - so which slice of `preorder` is the left subtree's own preorder?
Which slice is the right subtree's? Write the actual slices for Example 1.

**4.** Now the leap of faith, same as #226: you have a smaller `preorder` and a
smaller `inorder` for the left subtree, and another pair for the right subtree.
What single assumption lets you stop thinking here? And what is the base case -
what should you return when the slices come back **empty**?

**5.** Why does the constraint "`preorder` and `inorder` consist of **unique**
values" matter? Imagine the value 3 appeared twice in `inorder` - what step of
your plan breaks?

**6.** Your plan says "find the root inside `inorder`". If you do that with
`.index()` on every recursive call, you rescan the list once **per node** -
what does that cost in total? You solved this exact lookup problem before, in
Two Sum. What did you build there, and how many times do you need to build it
here?

## Two routes - do both

**A - slices + `.index()`** *(get it working first)*
Direct translation of questions 1-4: pop the root off `preorder`, split
`inorder` around it with `.index()`, recurse on the two slice pairs.
Correct, and fine at n = 3000 - but every call **copies** its slices and
rescans with `.index()`, so it is `O(n^2)` time and `O(n^2)` space in the
worst case. Which tree *shape* is that worst case? (You met it in #104 - the
10k chain.)

**B - hash map + boundaries, no copies** *(the real solution)*
Build `{value: index}` over `inorder` **once** - your Two Sum move. Then never
slice: each recursive call receives boundary indices into the original lists
(or just walks a single shared position through `preorder` - think about why
the next unused preorder element is always the root of whatever subtree you
are currently building). `O(n)` time, `O(n)` space for the map plus `O(h)`
recursion stack.

Route B without route A first is how you get lost in index arithmetic. Route A
without route B misses the point of the problem.

In [5]:
# Definition for a binary tree node.
import pandas as pa
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def TreePreorder(self,lst):
        if self is None: return
        lst.append(self.val)
        if self.left:
            self.left.TreePreorder( lst)
        if self.right:
            self.right.TreePreorder(lst)

    def TreeInorder(self,lst):
        if self is None: return
        if self.left:
            self.left.TreeInorder( lst)
        lst.append(self.val)
        if self.right:
            self.right.TreeInorder(lst)

# Level 4
n8 = TreeNode(8)
n9 = TreeNode(9)
n10 = TreeNode(10)
n11 = TreeNode(11)
n12 = TreeNode(12)
n13 = TreeNode(13)
n14 = TreeNode(14)
n15 = TreeNode(15)

# Level 3
n4 = TreeNode(4, n8, n9)
n5 = TreeNode(5, n10, n11)
n6 = TreeNode(6, n12, n13)
n7 = TreeNode(7, n14, n15)

# Level 2
n2 = TreeNode(2, n4, n5)
n3 = TreeNode(3, n6, n7)

# Root
root = TreeNode(1, n2, n3)

lstP = []
root.TreePreorder(lstP)
lstI = []
root.TreeInorder(lstI)

print(lstI)
print(lstP)


[8, 4, 9, 2, 10, 5, 11, 1, 12, 6, 13, 3, 14, 7, 15]
[1, 2, 4, 8, 9, 5, 10, 11, 3, 6, 12, 13, 7, 14, 15]


hello


### A helper to *see* the result

You are building a tree, so the tests must check its **shape**, not just its
values - `level_order` from #226 is not enough here, because it skips missing
children and two differently-shaped trees can print the same. `to_list` turns
your tree back into the LeetCode level-order list **with** `None` holes (the
exact inverse of the `build` helper from #114). Run this cell; don't edit it.

In [15]:
from collections import deque


def to_list(root):
    """Tree -> LeetCode level-order list, None for missing children."""
    if root is None:
        return []
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:   # trim the trailing Nones
        out.pop()
    return out


In [13]:
class Solution:
    def buildTree(self, preorder: list, inorder: list) -> TreeNode:
        if len(preorder) == 0 : return None
        rootVal = preorder[0]
        root = TreeNode(rootVal)
        rootValIndx = inorder.index(rootVal)
        I1 = inorder[ : rootValIndx ]
        I2 = inorder[rootValIndx +1 : ]
        P1 = preorder[1 : rootValIndx +1]
        P2 = preorder[rootValIndx +1 : ]
        root.left = self.buildTree(P1, I1)
        root.right = self.buildTree(P2, I2)
        return root



inorder = [8,4,9,2,10,5,11,1,12,6,13,3,14,7,15]
preorder = [1, 2, 4, 8, 9, 5, 10, 11, 3, 6, 12, 13, 7, 14, 15]
lst = []
Solution().buildTree(preorder, inorder).TreeInorder(lst)
print(lst)



[8, 4, 9, 2, 10, 5, 11, 1, 12, 6, 13, 3, 14, 7, 15]


In [16]:
# tests
TESTS = [
    # (preorder,                inorder,                expected LeetCode list)
    ([3, 9, 20, 15, 7],        [9, 3, 15, 20, 7],      [3, 9, 20, None, None, 15, 7]),
    ([-1],                     [-1],                   [-1]),
    ([1, 2, 4, 5, 3, 6, 7],    [4, 2, 5, 1, 6, 3, 7],  [1, 2, 3, 4, 5, 6, 7]),
    # the three shapes that all *print* alike but ARE not alike:
    ([1, 2, 3],                [3, 2, 1],              [1, 2, None, 3]),            # left chain
    ([1, 2, 3],                [1, 2, 3],              [1, None, 2, None, 3]),      # right chain
    ([1, 2, 3],                [2, 3, 1],              [1, 2, None, None, 3]),      # zigzag: left then right
]

for preorder, inorder, expected in TESTS:
    got = to_list(Solution().buildTree(preorder, inorder))
    print(f"{str(preorder):24} {str(inorder):24} -> {str(got):28} "
          f"{'OK' if got == expected else 'FAIL, want ' + str(expected)}")


[3, 9, 20, 15, 7]        [9, 3, 15, 20, 7]        -> [3, 9, 20, None, None, 15, 7] OK
[-1]                     [-1]                     -> [-1]                         OK
[1, 2, 4, 5, 3, 6, 7]    [4, 2, 5, 1, 6, 3, 7]    -> [1, 2, 3, 4, 5, 6, 7]        OK
[1, 2, 3]                [3, 2, 1]                -> [1, 2, None, 3]              OK
[1, 2, 3]                [1, 2, 3]                -> [1, None, 2, None, 3]        OK
[1, 2, 3]                [2, 3, 1]                -> [1, 2, None, None, 3]        OK


In [17]:
# the worst-case shape from route A's analysis: a 3000-node left chain.
# Route A passes this too (n = 3000 is small) - but time both routes on it
# and watch the gap. This is the #104 10k-chain lesson in a new coat.
n = 3000
chain_pre = list(range(1, n + 1))       # [1, 2, ..., 3000]
chain_in  = list(range(n, 0, -1))       # [3000, ..., 2, 1]

import sys
sys.setrecursionlimit(10_000)           # depth is n here - why?

root = Solution().buildTree(chain_pre, chain_in)

# walk down the left spine and count - no printing 3000 lines
depth, cur = 0, root
while cur:
    depth += 1
    cur = cur.left
print(f"left-chain depth: {depth}  {'OK' if depth == n else 'FAIL'}")


left-chain depth: 3000  OK


## After it passes

- **Round-trip it.** Take the tree you built from Example 1, run your
  `preorderPrint` from #114 on it, and check you get `preorder` back. A build
  and a traversal that are inverses of each other is the cleanest proof both
  are right.
- **Could you do it from preorder alone?** Try to invent two *different* trees
  with the same preorder `[1, 2]`. That is why the problem hands you two
  traversals - one fixes the order you visit, the other fixes where the split
  is. Which pair would NOT be enough: preorder + postorder? inorder +
  postorder? (One of these is LeetCode #106, one is #889 - and one of the two
  needs an extra assumption. Which, and why?)
- **Write the costs in one line each**, like #114: route A time/space, route B
  time/space, and *where* route B's `O(n)` space actually lives (hint: it is
  not the slices anymore).

In [ ]:
rootVal = 1
I1 = inorder[ : inorder.index(rootVal)]
I2 = inorder[inorder.index(rootVal) +1 : ]
P1 = preorder[1 : inorder.index(rootVal)+1]
P2 = preorder[inorder.index(rootVal) +1 : ]
print(I1,I2)
print(P1,P2)